# ezmerce v2 — API(엔드포인트) 통합 테스트

1차 스펙 대비 **실제 HTTP 엔드포인트**를 라이브 Supabase에 대고 테스트합니다.

**동작 방식**
- `TestClient(app)` 으로 FastAPI를 실제 라우팅 (실 JWT 발급은 생략하고 `get_current_user` 만 역할별로 오버라이드 → RBAC 가드는 그대로 동작)
- DB는 **실 Supabase**(서비스키, `.env`). 즉 라우팅·RBAC·DB·트리거가 전부 진짜로 도는 통합 테스트
- FK 제약(`created_by`/`wholesaler_id`) 때문에 실제 auth 계정·도매업체·프로필을 **시드**한 뒤 테스트하고, 마지막 **정리(cleanup)** 셀에서 삭제

**⚠️ 먼저 할 것**
1. `_05` 마이그레이션(`migrations/2026-06-03_v2_core_05_platform_code_fn.sql`)을 Supabase SQL Editor에서 실행 (아래 0-1 셀이 확인해 줌). 안 하면 상품 등록/엑셀 업로드가 platform_code 발급에서 실패합니다.
2. 위에서 아래로 순서대로 실행. 마지막 **5. 정리** 셀은 생성한 테스트 데이터를 모두 지웁니다.

> 비고: 이미지 업로드는 '프론트 직접 Storage' 모델이라, 이 노트북은 매칭 로직만 테스트합니다(실파일 업로드 없이 매니페스트만 전달). 실제 Storage 버킷은 불필요.

## 0. 셋업 — TestClient + 실 Supabase + 역할 오버라이드 헬퍼

In [ ]:
from pathlib import Path
import os, sys, io, uuid

here = Path.cwd()
BACKEND = here if (here / "app").exists() else (here.parent if (here.parent / "app").exists() else here)
os.chdir(BACKEND); sys.path.insert(0, str(BACKEND))

from contextlib import contextmanager
from fastapi.testclient import TestClient
from app.main import app
from app.core.auth import get_current_user
from app.core.supabase import get_supabase
from app.schemas.auth import CurrentUser

c = TestClient(app)
sb = get_supabase()                      # service key → 실 Supabase
XLSX_CT = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"

@contextmanager
def as_user(**kw):
    # get_current_user 를 특정 계정/역할로 오버라이드 (실 JWT 발급 생략). RBAC 가드는 그대로 실행됨.
    app.dependency_overrides[get_current_user] = lambda: CurrentUser(**kw)
    try:
        yield
    finally:
        app.dependency_overrides.pop(get_current_user, None)

print("python :", sys.version.split()[0])
print("routes :", len([r for r in app.routes if getattr(r, "methods", None)]))
print("/health:", c.get("/health").json())

## 0-1. `_05` 적용 확인 — platform_code 시퀀스 RPC

In [ ]:
try:
    seq = sb.rpc("next_platform_seq", {}).execute().data
    print("✅ next_platform_seq ->", seq, "( _05 적용됨 )")
except Exception as e:
    print("❌ _05 미적용으로 보입니다. SQL Editor에서 _05 마이그레이션을 먼저 실행하세요.")
    print("   에러:", str(e)[:300])

## 0-2. 테스트 픽스처 시드 (실 Supabase)

FK 제약상 `products.created_by`(→profiles), `products.wholesaler_id`(→wholesalers) 는 실재 행이어야 합니다.
`NBTEST` 태그로 도매업체 1 + auth계정·프로필(도매직원/관리자)을 만들고, 마지막에 정리합니다.

> `sb.auth.admin.create_user` 응답 형태(`res.user.id`)는 gotrue 버전 의존 — 여기서 처음 라이브 검증됩니다.

In [ ]:
TAG = "NBTEST"
def _email(role): return f"nbtest-{role}-{uuid.uuid4().hex[:8]}@example.com"

def seed_auth_user(role):
    res = sb.auth.admin.create_user({"email": _email(role), "password": "nbtest-pw-12345", "email_confirm": True})
    return res.user.id

# 도매업체
WS_ID = sb.table("wholesalers").insert({"name": f"{TAG}-도매", "biz_number": "000-00-00000"}).execute().data[0]["id"]
# 도매 직원(승인) = auth 계정 + profile
WS_UID = seed_auth_user("wholesaler")
sb.table("profiles").insert({"id": WS_UID, "role": "wholesaler", "status": "approved", "wholesaler_id": WS_ID}).execute()
# 관리자
ADMIN_UID = seed_auth_user("admin")
sb.table("profiles").insert({"id": ADMIN_UID, "role": "admin", "status": "approved"}).execute()

wholesaler_user = dict(id=WS_UID, role="wholesaler", status="approved", wholesaler_id=WS_ID)
admin_user      = dict(id=ADMIN_UID, role="admin", status="approved")
created_pids = []   # 정리용
print("WS_ID:", WS_ID, "\nWS_UID:", WS_UID, "\nADMIN_UID:", ADMIN_UID)

## 1. 권한 관리 — 회원가입 + 승인 (FR-1) · 스펙 ③ 계정분리/RBAC

- `POST /auth/register` (공개) — 자가가입 셀러는 `status=pending`, `price_visibility` 시드
- `GET /admin/accounts?status=pending` → `POST .../approve` → `POST .../price-visibility` (관리자)
- 권한상승 차단: 셀러가 `role=admin` 자가가입 → **400**

In [ ]:
reg = c.post("/auth/register", json={"email": _email("retail"), "password": "nbtest-pw-12345",
                                     "role": "retail_seller", "seller_type": "independent"})
print("register:", reg.status_code, reg.json())
SELLER_UID = reg.json()["id"]

with as_user(**admin_user):
    pend = c.get("/admin/accounts", params={"status": "pending"}).json()
    print("내 가입건 pending에 보임:", any(p["id"] == SELLER_UID for p in pend))
    print("approve     :", c.post(f"/admin/accounts/{SELLER_UID}/approve").status_code)
    print("price-vis    :", c.post(f"/admin/accounts/{SELLER_UID}/price-visibility",
                                   json={"price_visibility": "wholesale"}).status_code)

bad = c.post("/auth/register", json={"email": _email("x"), "password": "nbtest-pw-12345", "role": "admin"})
print("admin 자가가입 거부 (기대 400):", bad.status_code)

seller_user = dict(id=SELLER_UID, role="retail_seller", status="approved",
                   seller_type="independent", price_visibility="wholesale")

## 2. 상품 관리 CRUD (FR-2.1/2.4) · 스펙 ① 단건등록·수정·삭제·보관

- `POST /products` — 단건 등록(품번 정규화 → `platform_code` 발급)
- `PATCH /products/{id}` — 수정 / 보관(`status=archived`)
- `DELETE /products/{id}` — soft delete (`deleted_at`, 자식 cascade는 트리거)

In [ ]:
with as_user(**wholesaler_user):
    p = c.post("/products", json={"source_p_number": f"{TAG}-1001", "item_name": "NB 린넨셔츠",
        "skus": [{"color": "화이트", "size": "F", "wholesale_price": 12000, "retail_price": 29000}]})
    print("create:", p.status_code, "| platform_code:", p.json().get("platform_code"))
    PID = p.json()["id"]; CODE = p.json()["platform_code"]; created_pids.append(PID)

    print("수정 :", c.patch(f"/products/{PID}", json={"item_name": "NB 린넨셔츠(수정)"}).status_code)
    print("보관 :", c.patch(f"/products/{PID}", json={"status": "archived"}).status_code)
    print("복귀 :", c.patch(f"/products/{PID}", json={"status": "active"}).status_code)   # 카탈로그용으로 다시 active

    pdel = c.post("/products", json={"source_p_number": f"{TAG}-1009", "item_name": "NB 삭제용",
        "skus": [{"color": "블랙", "size": "F", "wholesale_price": 10000, "retail_price": 20000}]})
    PID_DEL = pdel.json()["id"]; created_pids.append(PID_DEL)
    print("soft delete:", c.delete(f"/products/{PID_DEL}").status_code, "(deleted_at 세팅)")

## 3. 대량 업로드 + 자동/수동 매칭 (FR-2.2/2.3) · 스펙 ① Bulk + 매칭

- `POST /uploads/excel` (multipart, **표준 템플릿**) → 품번별 상품 일괄생성 + `upload_jobs` 기록
- `POST /uploads/images` (프론트 Storage 매니페스트) → 파일명→품번 **자동 매칭**
- `GET /uploads/{job}/unmatched` → `POST /uploads/{job}/match` (수작업 매칭)

In [ ]:
import openpyxl
def xlsx_bytes(rows):
    wb = openpyxl.Workbook(); ws = wb.active
    ws.append(["품번", "상품명", "색상", "사이즈", "도매가", "판매가"])
    for r in rows: ws.append(r)
    b = io.BytesIO(); wb.save(b); return b.getvalue()

with as_user(**wholesaler_user):
    data = xlsx_bytes([
        (f"{TAG}-2001", "NB 데님", "인디고", "M", 20000, 45000),
        (f"{TAG}-2001", "NB 데님", "인디고", "L", 20000, 45000),   # 같은 품번 → 같은 상품, sku 추가
        (f"{TAG}-2002", "NB 코트", "카멜", "F", 50000, 99000),
    ])
    up = c.post("/uploads/excel", files={"file": ("pos.xlsx", data, XLSX_CT)}).json()
    print("excel: job", up["job_id"], "| 생성", len(up["created"]), "건 | 에러", len(up["errors"]))
    JOB = up["job_id"]
    created_pids += [p["id"] for p in up["created"]]

    img = c.post("/uploads/images", json={"job_id": JOB, "images": [
        {"original_filename": f"{TAG}-2001_front.jpg", "storage_path": f"{WS_ID}/{TAG}-2001_front.jpg"},
        {"original_filename": "정체불명.jpg",          "storage_path": f"{WS_ID}/정체불명.jpg"},
    ]}).json()
    print("images: matched", img["matched"], "| unmatched", img["unmatched"])

    un = c.get(f"/uploads/{JOB}/unmatched").json()
    print("unmatched 목록:", len(un), "건")
    if un:
        m = c.post(f"/uploads/{JOB}/match", json={"image_id": un[0]["id"], "source_p_number": f"{TAG}-2002"})
        print("수동 매칭:", m.status_code, "->", m.json().get("match_status"))

## 4. 폐쇄형 카탈로그 + 엑셀 + QR (FR-3/4/5) · 스펙 ②

- `GET /catalog` — 승인 셀러만, 역할별 가격 셰이핑(independent 셀러 → 도매가)
- `GET /catalog/export.xlsx` — 최우측 QR 열 삽입 엑셀
- `GET /qr/{code}.png` — QR 이미지 / `GET /p/{code}` — 공개 카드(가격 미포함)
- 미승인 셀러 → **403**

In [ ]:
with as_user(**seller_user):
    cat = c.get("/catalog").json()
    print("카탈로그 items:", len(cat["items"]))
    if cat["items"]:
        print("  예시 sku 가격(도매가 노출):", cat["items"][0]["skus"][0])
    xls = c.get("/catalog/export.xlsx")
    Path("/tmp/nb_catalog.xlsx").write_bytes(xls.content)
    print("export.xlsx:", len(xls.content), "bytes |", xls.headers.get("content-type"))

# 공개 엔드포인트(인증 불필요)
qr = c.get(f"/qr/{CODE}.png")
print("QR png:", len(qr.content), "bytes |", qr.headers["content-type"])
card = c.get(f"/p/{CODE}")
print("공개 카드:", card.json(), "| 가격필드 없음:", "price" not in card.json())

with as_user(id="pend", role="retail_seller", status="pending", seller_type="independent"):
    print("미승인 셀러 카탈로그 (기대 403):", c.get("/catalog").status_code)

### 4-1. QR 이미지 직접 보기

In [ ]:
from IPython.display import Image, display
display(Image(data=c.get(f"/qr/{CODE}.png").content))
print("스캔 대상 URL:", f"{CODE} → 공개 카드 /p/{CODE}")

## 5. 정리 (테스트 데이터 삭제)

테스트 픽스처는 **hard delete** 로 청소합니다(운영 soft-delete 규칙의 예외 — 테스트 정리용).
auth 계정 삭제 시 `profiles` 는 `ON DELETE CASCADE` 로 함께 사라집니다.

In [ ]:
# 상품/이미지/잡 (products 삭제 시 skus/images 는 FK cascade)
sb.table("product_images").delete().eq("wholesaler_id", WS_ID).execute()
sb.table("products").delete().eq("wholesaler_id", WS_ID).execute()
sb.table("upload_jobs").delete().eq("wholesaler_id", WS_ID).execute()

# 계정: 셀러 → 도매직원 → 관리자 순(approved_by FK 참조 때문에 셀러 먼저)
for uid in [SELLER_UID, WS_UID, ADMIN_UID]:
    try:
        sb.auth.admin.delete_user(uid)            # profiles ON DELETE CASCADE
    except Exception as e:
        print("delete_user", uid, "->", str(e)[:120])

sb.table("wholesalers").delete().eq("id", WS_ID).execute()
print("✅ cleanup 완료 — NBTEST 데이터 제거됨")

---
### 부록 — 1차 스펙 ↔ 엔드포인트 매핑

| 1차 스펙 | 엔드포인트 | 비고 |
|---|---|---|
| Bulk 업로드(엑셀+이미지) | `POST /uploads/excel`, `/uploads/images` | 엑셀은 **표준 템플릿** 전제 |
| 비정형 엑셀 그대로 파싱 | — | **현재 1차 비범위**(requirements §비범위). 표준 템플릿만 |
| 엑셀↔이미지 자동매칭 | `POST /uploads/images` | 파일명→품번 토큰 |
| 수작업 매칭 | `GET /uploads/{job}/unmatched`, `POST /uploads/{job}/match` | |
| 단건 등록/수정/삭제/보관 | `POST /products`, `PATCH`, `DELETE`(soft), `PATCH status=archived` | 보관=PATCH status |
| 폐쇄형 카탈로그 | `GET /catalog` | 승인 가드 + 가격 셰이핑 |
| 셀러 엑셀 다운로드(+QR열) | `GET /catalog/export.xlsx` | 최우측 QR |
| QR 생성 / 공개 카드 | `GET /qr/{code}.png`, `GET /p/{code}` | 카드 UI는 프론트 |
| 계정 3그룹 RBAC | `/auth/register`, `/admin/accounts/*` | role 화이트리스트 |
| 품번 정규화 | `platform_code`(SEQUENCE) | `_05` RPC 필요 |